# Valid failure overview

Looks across MNIST, CelebA, and SGSM. This is the honest overview table: pass rate only matters together with precondition match.

In [ ]:
from pathlib import Path
import csv, subprocess

REPO = 'https://github.com/casparbreloh/rbt4dnn-seminar.git'
DATA = Path('data/input.csv')
roots = [Path.cwd(), *Path.cwd().parents]

path = next((root / DATA for root in roots if (root / DATA).exists()), None)
if path is None:
    repo = Path('/content/rbt4dnn-seminar')
    if not repo.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO, str(repo)], check=True)
    path = repo / DATA

if not path.exists():
    raise FileNotFoundError(f'Could not find {DATA}')

ROOT = path.parents[1]
with path.open(newline='') as f:
    rows = list(csv.DictReader(f))

print(path)


In [1]:
def f(x):
    return None if x == '' else float(x)

def write_csv(path, fieldnames, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', newline='') as out:
        writer = csv.DictWriter(out, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

out_rows = []
for r in rows:
    p = f(r['pass_rate_mean'])
    m = f(r['precondition_match_mean'])
    n = int(r['n_images']) if r['n_images'] else None
    if p is None or m is None or n is None:
        continue
    out_rows.append({
        'dataset': r['dataset'],
        'requirement': r['requirement'],
        'variant': r['variant'],
        'n_images': n,
        'pass_rate': f'{p:.6f}',
        'precondition_match': f'{m:.6f}',
        'valid_failures': f'{n * m * (1 - p):.3f}',
        'nonmatching_failure_share': f'{(1 - m) * (1 - p):.6f}',
    })
out_rows.sort(key=lambda row: float(row['valid_failures']), reverse=True)

out = ROOT / 'data' / 'results' / 'valid-failures.csv'
fields = [
    'dataset', 'requirement', 'variant', 'n_images', 'pass_rate',
    'precondition_match', 'valid_failures', 'nonmatching_failure_share',
]
write_csv(out, fields, out_rows)

print('largest estimated valid-failure yields')
print('dataset req variant | pass | match | valid failures | nonmatch-fail share')
for r in out_rows[:12]:
    print(
        f"{r['dataset']} {r['requirement']} {r['variant']} | "
        f"{float(r['pass_rate']):.3f} | {float(r['precondition_match']):.3f} | "
        f"{float(r['valid_failures']):.1f} | {float(r['nonmatching_failure_share']):.3f}"
    )
print(out)


largest estimated valid-failure yields
dataset req variant | pass | match | valid failures | fp estimate
sgsm S2 lr | 0.134 | 0.850 | 7364.4 | 0.130
mnist M3 lr | 0.694 | 0.958 | 2930.5 | 0.013
sgsm S1 lr | 0.653 | 0.760 | 2633.4 | 0.083
celeba C6 lr | 0.796 | 0.981 | 2004.2 | 0.004
celeba C2 lr | 0.879 | 0.964 | 1170.3 | 0.004
sgsm S3 lr | 0.887 | 0.783 | 884.8 | 0.025
celeba C3 lr | 0.914 | 0.975 | 837.5 | 0.002
celeba C4 lr | 0.909 | 0.894 | 813.5 | 0.010
celeba C1 lr | 0.916 | 0.961 | 809.2 | 0.003
mnist M2 lr | 0.972 | 0.980 | 270.5 | 0.001
celeba C5 lr | 0.975 | 0.955 | 240.7 | 0.001
mnist M6 lr | 0.977 | 0.957 | 217.2 | 0.001


Rows without precondition-match data, such as M7/C7 in the paper data, are intentionally excluded from computed valid-failure estimates.